## Testing Mistral performance on 120 samples

Setup, load prompt, data

In [4]:
# If needed:
# %pip install mistralai pandas openpyxl scikit-learn simpledorff

import os
import json
import time
import pandas as pd
import numpy as np

from dotenv import load_dotenv
from mistralai.client import Mistral

load_dotenv()

client = Mistral(
    api_key=os.environ["MISTRAL_API_KEY"]
)

import importlib
import frame_prompts

importlib.reload(frame_prompts)

# --------------------------------------------------
# CHOOSE PROMPT
# --------------------------------------------------

SYSTEM_PROMPT = frame_prompts.OPERATIONALIZED_JOINT_PROMPT

PROMPT_NAME = "operationalized_joint"


# --------------------------------------------------
# MISTRAL SETTINGS
# --------------------------------------------------

MODEL = "mistral-large-latest"

client = Mistral(
    api_key=os.environ["MISTRAL_API_KEY"]
)


# --------------------------------------------------
# LOAD GOLD DATA
# --------------------------------------------------

df = pd.read_excel("frame_gold.xlsx")

required_columns = [
    "uuid",
    "title",
    "protest_excerpt",
    "title_frame",
    "excerpt_frame",
    "demands_present",
    "protester_quote",
    "authority_quote",
    "thematic_present",
]

missing = [c for c in required_columns if c not in df.columns]

if missing:
    raise ValueError(f"Missing columns: {missing}")

df = df[required_columns].copy()

df["title"] = df["title"].fillna("").astype(str)
df["protest_excerpt"] = df["protest_excerpt"].fillna("").astype(str)


# --------------------------------------------------
# NORMALIZE GOLD LABELS
# --------------------------------------------------

label_columns = [
    "title_frame",
    "excerpt_frame",
    "demands_present",
    "protester_quote",
    "authority_quote",
    "thematic_present",
]

for col in label_columns:
    df[col] = (
        df[col]
        .astype(str)
        .str.strip()
        .str.lower()
        .str.replace(" ", "_", regex=False)
    )

df["title_frame"] = df["title_frame"].replace(
    "violence",
    "protester_violence"
)

df["excerpt_frame"] = df["excerpt_frame"].replace(
    "violence",
    "protester_violence"
)


# --------------------------------------------------
# TEST FIRST
# --------------------------------------------------

test_df = df.head(3).copy()

RUN_DF = df

# When ready:
# RUN_DF = df

print(f"Running {len(RUN_DF)} samples.")
print(f"Model: {MODEL}")
print(f"Prompt: {PROMPT_NAME}")

Running 120 samples.
Model: mistral-large-latest
Prompt: operationalized_joint


schema + API call + run

In [6]:
# ============================================================
# VALID LABELS
# ============================================================

FRAME_LABELS = [
    "protester_violence",
    "harm_to_protesters",
    "property_damage",
    "disruption",
    "grievance",
    "widespread_support",
    "peaceful",
    "impactful",
    "residual",
]

BINARY_LABELS = ["yes", "no"]


# ============================================================
# FORMAT INPUT
# ============================================================

def format_input(title, excerpt):
    return f"""TITLE:
{title}

EXCERPT:
{excerpt}
"""


# ============================================================
# VALIDATE MODEL OUTPUT
# ============================================================

def validate_annotation(annotation):

    required_keys = {
        "title_frame",
        "excerpt_frame",
        "demands_present",
        "protester_quote",
        "authority_quote",
        "thematic_present",
    }

    if set(annotation.keys()) != required_keys:
        raise ValueError(
            f"Unexpected JSON keys: {annotation.keys()}"
        )

    if annotation["title_frame"] not in FRAME_LABELS:
        raise ValueError(
            f"Invalid title_frame: {annotation['title_frame']}"
        )

    if annotation["excerpt_frame"] not in FRAME_LABELS:
        raise ValueError(
            f"Invalid excerpt_frame: {annotation['excerpt_frame']}"
        )

    for col in [
        "demands_present",
        "protester_quote",
        "authority_quote",
        "thematic_present",
    ]:
        if annotation[col] not in BINARY_LABELS:
            raise ValueError(
                f"Invalid {col}: {annotation[col]}"
            )

    return annotation


# ============================================================
# CALL MISTRAL
# ============================================================

def annotate_sample(title, excerpt, max_retries=3):

    messages = [
        {
            "role": "system",
            "content": SYSTEM_PROMPT,
        },
        {
            "role": "user",
            "content": format_input(title, excerpt),
        },
    ]

    for attempt in range(max_retries):

        try:

            response = client.chat.complete(
                model=MODEL,
                messages=messages,

                # JSON mode ensures valid JSON.
                response_format={
                    "type": "json_object"
                },

                temperature=0,
            )

            raw_text = response.choices[0].message.content

            annotation = json.loads(raw_text)

            annotation = validate_annotation(annotation)

            # Usage fields may vary slightly by SDK version,
            # so access them defensively.
            input_tokens = getattr(
                response.usage,
                "prompt_tokens",
                None
            )

            output_tokens = getattr(
                response.usage,
                "completion_tokens",
                None
            )

            return {
                **annotation,
                "input_tokens": input_tokens,
                "output_tokens": output_tokens,
                "raw_response": raw_text,
                "error": None,
            }

        except Exception as e:

            print(
                f"Attempt {attempt + 1}/{max_retries} failed: "
                f"{type(e).__name__}: {e}"
            )

            if attempt < max_retries - 1:
                time.sleep(2 ** attempt)

            else:

                return {
                    "title_frame": None,
                    "excerpt_frame": None,
                    "demands_present": None,
                    "protester_quote": None,
                    "authority_quote": None,
                    "thematic_present": None,
                    "input_tokens": None,
                    "output_tokens": None,
                    "raw_response": None,
                    "error": str(e),
                }


# ============================================================
# RUN
# ============================================================

results = []

for count, (_, row) in enumerate(
    RUN_DF.iterrows(),
    start=1
):

    print(
        f"[{count}/{len(RUN_DF)}] "
        f"{row['uuid']}"
    )

    prediction = annotate_sample(
        title=row["title"],
        excerpt=row["protest_excerpt"],
    )

    prediction["uuid"] = row["uuid"]

    results.append(prediction)
    time.sleep(15)


pred_df = pd.DataFrame(results)


# ============================================================
# RENAME PREDICTIONS
# ============================================================

pred_df = pred_df.rename(columns={
    "title_frame": "pred_title_frame",
    "excerpt_frame": "pred_excerpt_frame",
    "demands_present": "pred_demands_present",
    "protester_quote": "pred_protester_quote",
    "authority_quote": "pred_authority_quote",
    "thematic_present": "pred_thematic_present",
})


# ============================================================
# MERGE BY UUID
# ============================================================

results_df = RUN_DF.merge(
    pred_df,
    on="uuid",
    how="left",
    validate="one_to_one"
)


# ============================================================
# SAVE IMMEDIATELY
# ============================================================

output_file = (
    f"{PROMPT_NAME}_{MODEL}.xlsx"
)

results_df.to_excel(
    output_file,
    index=False
)

print(f"\nSaved: {output_file}")

print(
    f"Successful: "
    f"{results_df['error'].isna().sum()}"
)

print(
    f"Errors: "
    f"{results_df['error'].notna().sum()}"
)

if results_df["input_tokens"].notna().any():
    print(
        f"Average input tokens: "
        f"{results_df['input_tokens'].mean():,.0f}"
    )

if results_df["output_tokens"].notna().any():
    print(
        f"Average output tokens: "
        f"{results_df['output_tokens'].mean():,.0f}"
    )

display(results_df.head())

[1/120] d1334539-4cf3-4c5f-a0ef-88db5f167395
[2/120] b088e93f-1de4-4021-97bf-a90700b0a0ac
[3/120] fffb4a3a-8ee2-4cc4-9b9a-b0f676c5bdad
[4/120] 155faae5-c8b6-441e-a20e-dce06e01cf32
[5/120] 50fea669-6828-44de-8a70-b448e398be11
[6/120] 81efb265-e3ba-4508-adc7-73cbc029f359
[7/120] 5fb219a8-9df7-4ad0-bc57-d46be3e327e0
[8/120] 252fb33d-5e6c-4cff-acc0-98c253f6c4b8
[9/120] 4acf5304-e74d-4a4f-bee0-caf65bd02abe
[10/120] 74fd09e2-35a3-495b-9b59-1610ec3398e1
[11/120] 7d9d161f-8087-4e0b-89e2-4a9bc1dcfd9d
[12/120] 0c5fa89f-f32f-4014-98e5-e8ad481b8a2f
[13/120] b2a2ef91-319a-47bf-9a61-1f8b4e192abf
[14/120] a92ef5b3-9891-4492-8a32-5024d7cc52dc
[15/120] c13350d6-ae7e-43f9-8999-17e530a4c80b
[16/120] 874c16ec-4684-40c7-8928-06baaa20260d
[17/120] 0192cd22-a165-4eb8-bf6d-da6165164b2e
[18/120] 95c2c809-f17e-4d9d-af74-26ba987929cd
[19/120] 109cd735-179b-4d53-ba32-ac1b70d5d990
[20/120] 36ab3319-2965-4075-80bd-14627327ab65
[21/120] f4bf7615-7d65-4e3d-b332-fbf590087f99
[22/120] 58a57f20-02b7-4389-9add-4cd5c76aae

,uuid,title,protest_excerpt,title_frame,excerpt_frame,demands_present,protester_quote,authority_quote,thematic_present,pred_title_frame,pred_excerpt_frame,pred_demands_present,pred_protester_quote,pred_authority_quote,pred_thematic_present,input_tokens,output_tokens,raw_response,error
0,d1334539-4cf3-4c5f-a0ef-88db5f167395,Third politician in a month killed in Ecuador,The gang boss was moved out of a jail with lig...,residual,grievance,yes,no,no,no,residual,grievance,yes,yes,no,no,4528,64,"{\n ""title_frame"": ""residual"",\n ""excerpt_fr...",None
1,b088e93f-1de4-4021-97bf-a90700b0a0ac,Amazon CEO tells staff to work in office three...,This comes after Amazon said it would lay off ...,residual,widespread_support,yes,yes,no,yes,residual,grievance,yes,yes,no,no,4601,64,"{\n ""title_frame"": ""residual"",\n ""excerpt_fr...",None
2,fffb4a3a-8ee2-4cc4-9b9a-b0f676c5bdad,Gaza ceasefire protest in Capitol rotunda lead...,Capitol rotunda on Sept. 27. Photo: Michael Ro...,grievance,disruption,yes,no,yes,yes,disruption,disruption,yes,no,yes,no,4592,63,"{\n ""title_frame"": ""disruption"",\n ""excerpt_...",None
3,155faae5-c8b6-441e-a20e-dce06e01cf32,"Daniel Penny, accused in Jordan Neely subway c...",Neely and could not have foreseen his untimely...,residual,grievance,yes,yes,yes,no,residual,harm_to_protesters,yes,no,yes,no,4528,66,"{\n ""title_frame"": ""residual"",\n ""excerpt_fr...",None
4,50fea669-6828-44de-8a70-b448e398be11,Legal scholars raising constitutional argument...,Not all in the legal community agree – and wha...,residual,protester_violence,no,no,yes,yes,residual,grievance,yes,no,yes,yes,4845,64,"{\n ""title_frame"": ""residual"",\n ""excerpt_fr...",None


summary and results

In [7]:
# ============================================================
# EVALUATE MISTRAL AGAINST HUMAN GOLD
# ============================================================

from sklearn.metrics import (
    accuracy_score,
    f1_score,
    precision_score,
    recall_score,
)

import simpledorff


# ------------------------------------------------------------
# KRIPPENDORFF ALPHA
# ------------------------------------------------------------

def calculate_alpha(y_true, y_pred):

    alpha_df = pd.DataFrame({
        "item_id": np.arange(len(y_true)).repeat(2),
        "annotator_id": ["human", "model"] * len(y_true),
        "label": np.column_stack([
            y_true.to_numpy(),
            y_pred.to_numpy()
        ]).reshape(-1)
    })

    return simpledorff.calculate_krippendorffs_alpha_for_df(
        alpha_df,
        experiment_col="item_id",
        annotator_col="annotator_id",
        class_col="label"
    )


# ------------------------------------------------------------
# TASKS
# ------------------------------------------------------------

tasks = {
    "title_frame": {
        "gold": "title_frame",
        "pred": "pred_title_frame",
        "type": "frame",
    },

    "excerpt_frame": {
        "gold": "excerpt_frame",
        "pred": "pred_excerpt_frame",
        "type": "frame",
    },

    "demands_present": {
        "gold": "demands_present",
        "pred": "pred_demands_present",
        "type": "binary",
    },

    "protester_quote": {
        "gold": "protester_quote",
        "pred": "pred_protester_quote",
        "type": "binary",
    },

    "authority_quote": {
        "gold": "authority_quote",
        "pred": "pred_authority_quote",
        "type": "binary",
    },

    "thematic_present": {
        "gold": "thematic_present",
        "pred": "pred_thematic_present",
        "type": "binary",
    },
}


# ------------------------------------------------------------
# METRICS
# ------------------------------------------------------------

summary_rows = []

for task_name, info in tasks.items():

    valid = results_df[
        [info["gold"], info["pred"]]
    ].dropna()

    y_true = valid[info["gold"]]
    y_pred = valid[info["pred"]]

    row = {
        "task": task_name,
        "n": len(valid),

        "accuracy": accuracy_score(
            y_true,
            y_pred
        ),

        "macro_f1": f1_score(
            y_true,
            y_pred,
            average="macro",
            zero_division=0
        ),

        "kripp_alpha": calculate_alpha(
            y_true,
            y_pred
        ),
    }


    if info["type"] == "frame":

        row["weighted_f1"] = f1_score(
            y_true,
            y_pred,
            average="weighted",
            zero_division=0
        )

        row["yes_precision"] = np.nan
        row["yes_recall"] = np.nan
        row["yes_f1"] = np.nan


    else:

        row["weighted_f1"] = np.nan

        row["yes_precision"] = precision_score(
            y_true,
            y_pred,
            pos_label="yes",
            zero_division=0
        )

        row["yes_recall"] = recall_score(
            y_true,
            y_pred,
            pos_label="yes",
            zero_division=0
        )

        row["yes_f1"] = f1_score(
            y_true,
            y_pred,
            pos_label="yes",
            zero_division=0
        )


    summary_rows.append(row)


summary_df = pd.DataFrame(summary_rows)

metric_cols = [
    "accuracy",
    "macro_f1",
    "yes_precision",
    "yes_recall",
    "yes_f1",
    "weighted_f1",
    "kripp_alpha",
]

summary_df[metric_cols] = (
    summary_df[metric_cols]
    .round(3)
)


display(summary_df)


# ------------------------------------------------------------
# SAVE SUMMARY
# ------------------------------------------------------------

summary_file = (
    f"summary_"
    f"{PROMPT_NAME}_{MODEL}.xlsx"
)

summary_df.to_excel(
    summary_file,
    index=False
)

print(f"Saved: {summary_file}")

,task,n,accuracy,macro_f1,kripp_alpha,weighted_f1,yes_precision,yes_recall,yes_f1
0,title_frame,120,0.700,0.570,0.492,0.725,NaN,NaN,NaN
1,excerpt_frame,120,0.508,0.439,0.392,0.478,NaN,NaN,NaN
2,demands_present,120,0.733,0.683,0.368,NaN,0.800,0.819,0.810
3,protester_quote,120,0.850,0.847,0.694,NaN,0.824,0.824,0.824
4,authority_quote,120,0.800,0.795,0.592,NaN,0.905,0.760,0.826
5,thematic_present,120,0.808,0.807,0.615,NaN,0.754,0.827,0.789


Saved: summary_operationalized_joint_mistral-large-latest.xlsx
